# RAG - Retrieval Augmented Generations    
* Is a patern that can improve the effecacy of a LLM application by leveraging custom data   
* Is done by retieving data/documents relevants to a question or task and providing them as context to augment the prompts to a LLM to improve generation.

#### RAG Use Cases    

* Q&A Chatbots   
* Search Augmentation   
* Content Criation and Summarization

#### Main concepts of RAG Workflow   
**1.  Index and Embed**: An embedding model used to creating vector representation of the documents and users queries.   
**2.  Vector store**: Specialized to store unstructured data indexed by vectros. Vectors can be sotred with a **vector DB**, **library** or **plugin**   
**3.  Retrieval**: Search stored vectors uing similairity search to efficiently retrieve relavant information   
**4.  Filtering & Reranking**: The process of selectting or raking retrieved documents before passing as context. Filtering can be **pre-, in-, post-query**    
**5.  Prompt Augmentation**: Prompt engineering workflow to enhace context via injections of data retrieved from Vector store    
**6.  Generation**: A LLM used for generating a response for the user's request. 

#### Benefits of RAG Architecture
* Up-to-date and accurate response    
* Reducing inaccurate response or hallucination    
* Damin-specific contextitualization   
* Efficiency and cost0effectiveness

In [0]:
%pip install mlflow==2.10.1 lxml==4.9.3 langchain==0.1.5 databricks-vectorsearch==0.22 cloudpickle==2.2.1 databricks-sdk==0.18.0 cloudpickle==2.2.1 pydantic==2.5.2 openpyxl
%pip install pip mlflow[databricks]==2.10.1

dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
import pandas as pd
import random
import os
import uuid
import json
import io
import plotly.express as px
from langchain_community.chat_models import ChatDatabricks

from datetime import datetime
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType

In [0]:
chat_endpoint = "databricks-llama-4-maverick" 
chat_model = ChatDatabricks(endpoint = chat_endpoint,
                            temperature = 0,
                            max_tokens = 8000) 

In [0]:
print(chat_model.invoke("Crie um texto sobre um robô que não gosta de  trabalhar aos finais de semana. Depois utilize o texto e elabore uma questão de multipla escolha sobre a interpretação do texto.").content)

In [0]:
system_prompt = """Atue como um professor de ensino fundamental brasileiro. Você é um especialista em formular questões para avaliar os alunos. Com sua vasta experiência, suas questões são sempre claras, objetivas e respeitam o nível de aluno."""

input_prompt = """15 questões de múltipla escolha, sendo 5 de matemática e 10 de língua portuguesa.

# Instruções para a prova de matemática
1 - A prova de matemática deve ser com questões de situações problema. Como nos exemplos abaixo, e com mais de uma etapta de raciocínio.

Exemplo 1:
Um vendedor de maçãs tem 120 maçãs. Ele vendeu 2/3 para seu amigo. Após a venda, ele comprou mais 30 maçãs. Quantas maças ele ficou.

Exemplo 2:
Um prédio tem 120 andares, cada andar tem 4 apartamentos e cada apartamento tem 10 portas. Quantas portas há no total no prédio?

Exemplo 3:
João tem R$ 75,00. Ele comprou um livro por R$ 25,00 e 1 par de chinelos por R$ 32,00. Depois sua mãe, lhe deu mais R$ 22,00. Com quantos reais ele ficou?

Utilize esses exemplos apenas para criar questões parecidas. Não as reproduza no simulado.


# Instruções para a prova de português

1 - Das 10 questões presentes no simulado de língua portuguesa, cinco delas deve ser sobre interpretação de textos e as outras 20 com os assuntos dos conteúdos mostrados abaixo.

Para as questões de multipla escolha crie textos de um parágrafo de 6 linhas para formular as questões. Em seguida gere uma questão sobre o texto. Como mostrado nos exemplos abaixo.

Exemplos de questões de interpretação de texto:
Exemplo 1: 
Enunciado:
    Leia o texto: 'O menino comeu um sanduíche. Ele estava com fome.' Qual é a relação entre as duas orações?
Alternativas: 
    Causa e efeito
    Contraste
    Adição
    Consequência

Exemplo 2: 
Enunciado:
    Leia o texto: 'O menino sorri ao vender pirulitos coloridos na praia. 
    Ele oferece seus doces para as crianças que brincam na areia. 
    Seu carrinho é cheio de pirulitos de todos os sabores. 
    As pessoas compram seus pirulitos e elogiam o seu sorriso. 
    Ele é feliz vendendo pirulitos e fazendo as pessoas felizes.
    Qual é o sentimento do menino enquanto vende pirulitos na praia?
Alternativas: 
    Tristeza, pois ele está trabalhando.
    Indiferença, pois ele não se importa com o que está fazendo.
    Felicidade, pois ele gosta de vender pirulitos e fazer as pessoas felizes.
    Raiva, pois as pessoas não estão comprando seus pirulitos.' Qual é a relação entre as duas orações?

Exemplo 3: 
Enunciado:
    Leia o texto: 'Zeta era um robô projetado para realizar uma variedade de tarefas em uma fábrica moderna. Ele era eficiente, rápido e sempre pronto para trabalhar. No entanto, Zeta tinha uma peculiaridade: ele detestava trabalhar aos finais de semana.
    Enquanto seus colegas robôs pareciam não se importar com o dia da semana, Zeta sempre se sentia um pouco triste quando via o calendário indicar sábado ou domingo. Ele sonhava em ter esses dias livres para "recarregar suas baterias" e realizar atividades que nada tivessem a ver com parafusos, soldas ou montagens.
    Certo sábado, enquanto realizava suas tarefas rotineiras com um suspiro digital, Zeta confidenciou a um de seus colegas: "Se eu fosse programado para ter finais de semana de folga, acho que seria o robô mais feliz da fábrica." Seu colega, um robô mais antigo e sábio, respondeu: "Talvez um dia, Zeta. Talvez um dia.' Com base no texto, o que Zeta gostaria de ter nos finais de semana?
Alternativas: 
    Mais tarefas para realizar.
    Fins de semana de folga para descansar e fazer coisas diferentes.
    Uma programação mais complexa.
    Uma bateria mais potente


# Conteúdos que devem constar nas questões.

Para elaborar as questões considere os conteúdos abaixo.

Conteúdos de língua Portuguesa:
* Leitura e Interpretação de texto;
* Classes Gramaticais: Substantivo, Artigo, Adjetivo, Pronome, Advérbio, Numeral, Interjeição, Preposição,
Conjunção e Verbo;
* Formação e Significação de palavras;
* Acentuação;
* Pontuação;
* Ortografia;
* Variação Linguística;
* Discurso Direto e Discurso Indireto.

Conteúdos de Matemática:
* Sistema de Numeração Decimal;
* Múltiplos;
* Unidades de Medida (capacidade, volume e tempo);
* Fração;
* Número Decimal;
* Probabilidade;
* Sistema Monetário Brasileiro;
* Área e Perímetro;
* Situação-problema (raciocínio);
* Poliedro e Corpo;
* Polígono;
* Ângulo.

# Público alvo
O nível do simulado dever considerar alunos de dez anos de idade que estejam no quinto ano do ensino fundamental brasileiro.

# Formato do simulado
O simulado deve ser criado em formato json como no exemplo abaixo:

{"língua portuguesa": questões:[
{
"questao 1": <Enunciado da questão 1>,
"alternativas": <quatro alternativas de resposta para a questão 1>,
"gabarito": <alternativa correta da questão 1>,
"dica": <Dica de solução da questão 1>,
"explicacao": <Explicação da questão 1>	
},
{"questao 2": <Enunciado da questão 2>,
"alternativas": <quatro alternativas de resposta para a questão 2>,
"gabarito": <alternativa correta da questão 2>,
"dica": <Dica de solução da questão 2>,
"explicacao": <Explicação da questão 2>
},
{"questao 3": <Enunciado da questão 3>,
"alternativas": <quatro alternativas de resposta para a questão 3>,
"gabarito": <alternativa correta da questão 3>,
"dica": <Dica de solução da questão 3>,
"explicacao": <Explicação da questão 3>
}
]
"matemática":questões:[
{
"questao 1": <Enunciado da questão 1>,
"alternativas": <quatro alternativas de resposta para a questão 1>,
"gabarito": <alternativa correta da questão 1>,
"dica": <Dica de solução da questão 1>,
"explicacao": <Explicação da questão 1>	
},
{"questao 2": <Enunciado da questão 2>,
"alternativas": <quatro alternativas de resposta para a questão 2>,
"gabarito": <alternativa correta da questão 2>,
"dica": <Dica de solução da questão 2>,
"explicacao": <Explicação da questão 2>
},
{"questao 3": <Enunciado da questão 3>,
"alternativas": <quatro alternativas de resposta para a questão 3>,
"gabarito": <alternativa correta da questão 3>,
"dica": <Dica de solução da questão 3>,
"explicacao": <Explicação da questão 3>
}]
}

Sua resposta deve ser somente o json criado no padrão <conteúdo do json'>
"""

questoes = chat_model.invoke(system_prompt + input_prompt).content

In [0]:
    };


In [0]:
system_prompt = """Atue como um analista de desenvolvimento senior. Você tem muito conhecimento sobre os padrões utilizados nos arquivos json"""

input_prompt = f"""Analise o arquivo json e o formate no padrão json correto.

Arquivo json: {json.dumps(questoes)}

Sua resposta deve ser apenas o json formatado no padrão <conteúdo do json>'
"""

questoes = chat_model.invoke(system_prompt + input_prompt).content

In [0]:

symtem_prompt = "Atue como um desenvolvedor full stack. Você é um programador senior e tem grande experiência em UI e UX."

input_prompt = f"""
Vou te fornecer um arquivo json contendo questões de matemática e lingua portuguesa. Crie uma página interativa para eu realizar um simulado de uma avaliação utilizando as questões presentes no arquivo json. Inclua todo o conteúdo do josn no html de modo que eu consiga realizar o simulado na integra.

REQUISITOS DA TELA:
- O Simulado deve conter todas as questões presentes no arquivo json.
- Enumere em ordem crescente;
- Mostre as questões uma por uma;
- A Tela deve mostrar somente o enunciado e as alternativas;
- Não destacar a alternativa correta;
- As explicações devem ficar ocultas e aparecer somente quando o usuário clicar em uma das alternativas;
- Abaixo das alternativas inclua um botão de dica. Caso o usuário clique neste botão, a dica para a solução da questão será mostrada;
- Abaixo do botão de dica inclua dois botões de navegação para avançar ou voltar de uma questão para outra;
- Os botões de navegação devem ser na cor verde e com os cantos arredondados e alinhados à esquerda;
- O botão de dica deve ser na cor azul clara e com os cantos arredondados e alinhado à esquerda;
- Mostre as alternativas em div de cor cinza claro e com cantos arredondados;
- Após a última questão do simulado, deve aparecer uma tela com a porcentagem de acertos e um botão com label "Revisão". Caso o usuário clique neste botão ele volta para a tela da primeira questão para revisar as suas respostas;


Arquivo json: {json.dumps(questoes.replace("```json", "").replace("```", ""))}

Utilize o conteúdo do arquivo json para criar a página interativa. Gere o código completo com toda a estrutura de um arquivo html.

ATENÇÂO: A pagina deve ser criada seguindo todos os requisitos apresentados.
Sua resposta deve ser somente o código html da página no padrão <conteúdo do html da página>. Não inclua nada que não seja conteúdo html
"""

html = chat_model.invoke(system_prompt + input_prompt).content

In [0]:
html = """<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Simulado de Avaliação</title>
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;600&display=swap');

        :root {
            --primary-color: #28a745;
            --secondary-color: #007bff;
            --background-color: #f0f2f5;
            --card-background: #ffffff;
            --option-background: #f8f9fa;
            --correct-color: #28a745;
            --incorrect-color: #dc3545;
            --hint-color: #007bff;
            --text-color: #333;
        }

        body {
            font-family: 'Poppins', sans-serif;
            background-color: var(--background-color);
            color: var(--text-color);
            margin: 0;
            padding: 20px;
            display: flex;
            justify-content: center;
            align-items: center;
            min-height: 100vh;
        }

        .container {
            width: 100%;
            max-width: 700px;
            background-color: var(--card-background);
            padding: 30px;
            border-radius: 15px;
            box-shadow: 0 4px 20px rgba(0, 0, 0, 0.1);
        }

        .question-container {
            display: none;
        }

        .question-container.active {
            display: block;
        }

        .header h1 {
            text-align: center;
            font-size: 2.5em;
            color: var(--primary-color);
            margin-bottom: 10px;
        }

        .question-nav {
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-top: 20px;
            margin-bottom: 20px;
        }

        .progress-bar {
            background-color: #e9ecef;
            border-radius: 5px;
            height: 10px;
            overflow: hidden;
            width: 100%;
        }

        .progress {
            height: 100%;
            background-color: var(--primary-color);
            transition: width 0.4s ease;
        }

        .question-info {
            font-size: 1.2em;
            font-weight: bold;
            color: #555;
        }

        .enunciado {
            font-size: 1.1em;
            line-height: 1.6;
            margin-top: 20px;
            margin-bottom: 25px;
            white-space: pre-wrap;
        }

        .alternativas {
            list-style-type: none;
            padding: 0;
            margin: 0;
        }

        .alternativa-item {
            cursor: pointer;
            padding: 15px;
            margin-bottom: 12px;
            border: 2px solid #ddd;
            border-radius: 10px;
            background-color: var(--option-background);
            transition: all 0.3s ease;
            font-weight: 500;
        }

        .alternativa-item:hover {
            transform: translateY(-3px);
            box-shadow: 0 4px 10px rgba(0, 0, 0, 0.1);
        }

        .alternativa-item.correct {
            border-color: var(--correct-color);
            background-color: #d4edda;
            color: var(--correct-color);
        }

        .alternativa-item.incorrect {
            border-color: var(--incorrect-color);
            background-color: #f8d7da;
            color: var(--incorrect-color);
        }

        .explanation {
            display: none;
            margin-top: 20px;
            padding: 15px;
            background-color: #e9f5ff;
            border-left: 5px solid var(--hint-color);
            border-radius: 5px;
            line-height: 1.5;
        }

        .explanation h4 {
            margin-top: 0;
            color: var(--hint-color);
        }

        .hint-button {
            background-color: var(--hint-color);
            color: white;
            padding: 10px 20px;
            border: none;
            border-radius: 20px;
            cursor: pointer;
            margin-top: 20px;
            font-size: 1em;
            align-self: flex-start;
        }
        .hint-button:hover {
            background-color: #0056b3;
        }
        
        .hint-text {
            display: none;
            margin-top: 10px;
            padding: 10px;
            background-color: #f0f8ff;
            border-left: 4px solid var(--hint-color);
            border-radius: 5px;
        }

        .navigation-buttons {
            display: flex;
            justify-content: flex-start;
            margin-top: 20px;
        }

        .navigation-buttons button {
            background-color: var(--primary-color);
            color: white;
            padding: 10px 20px;
            border: none;
            border-radius: 20px;
            cursor: pointer;
            font-size: 1em;
            margin-right: 15px;
        }
        
        .navigation-buttons button:hover {
            background-color: #218838;
        }

        .navigation-buttons button:disabled {
            background-color: #ccc;
            cursor: not-allowed;
        }

        .results-container {
            display: none;
            text-align: center;
        }

        .results-container h2 {
            font-size: 2em;
            color: var(--primary-color);
        }

        .results-container p {
            font-size: 1.5em;
            margin: 15px 0;
        }
        
        .results-container button {
            background-color: var(--primary-color);
            color: white;
            padding: 10px 20px;
            border: none;
            border-radius: 20px;
            cursor: pointer;
            font-size: 1em;
        }
        
        .results-container button:hover {
            background-color: #218838;
        }

    </style>
</head>
<body>

<div class="container">
    <div id="quiz-container">
        <div class="header">
            <h1>Simulado de Avaliação</h1>
            <div class="progress-bar">
                <div class="progress" id="progress-bar"></div>
            </div>
        </div>

        <div id="questions-area"></div>
    </div>
    
    <div id="results-area" class="results-container">
        <h2>Simulado Concluído!</h2>
        <p>Você acertou <span id="correct-count"></span> de 50 questões.</p>
        <p>Sua porcentagem de acertos é: <span id="percentage"></span>%</p>
        <button id="review-button">Revisão</button>
    </div>
</div>

<script>
    const data = {
        "simulado": {
            "titulo": "Simulado de Avaliação - 5º Ano do Ensino Fundamental",
            "instrucoes": "Responda às questões de múltipla escolha. Apenas uma alternativa está correta em cada questão.",
            "questoes": [
      {
        "id": 1,
        "disciplina": "Matemática",
        "enunciado": "Numa escola, há 350 alunos. Desses, 2/5 praticam futebol. Do restante, 1/3 pratica vôlei. Quantos alunos praticam vôlei?",
        "alternativas": [
          "a) 70 alunos",
          "b) 140 alunos",
          "c) 120 alunos",
          "d) 105 alunos"
        ],
        "resposta_correta": "a) 70 alunos",
        "explicacao": "Primeiro, calculamos o número de alunos que praticam futebol: (2/5) * 350 = 140. O restante é 350 - 140 = 210. Desses, 1/3 pratica vôlei: (1/3) * 210 = 70. Portanto, 70 alunos praticam vôlei.",
        "dica": "Primeiro, calcule quantos alunos praticam futebol. Depois, subtraia esse número do total de alunos. Por fim, calcule 1/3 do valor que sobrou."
      },
      {
        "id": 2,
        "disciplina": "Matemática",
        "enunciado": "Uma caixa de laranjas tem 240 unidades. O fazendeiro vendeu 1/4 da caixa na feira e, depois, deu 1/6 do que sobrou para sua vizinha. Quantas laranjas sobraram na caixa?",
        "alternativas": [
          "a) 150 laranjas",
          "b) 180 laranjas",
          "c) 160 laranjas",
          "d) 144 laranjas"
        ],
        "resposta_correta": "a) 150 laranjas",
        "explicacao": "O fazendeiro vendeu (1/4) * 240 = 60 laranjas. Sobraram 240 - 60 = 180 laranjas. Ele deu (1/6) * 180 = 30 laranjas. Sobraram 180 - 30 = 150 laranjas. Portanto, sobraram 150 laranjas na caixa.",
        "dica": "Calcule a quantidade de laranjas vendidas primeiro. Subtraia esse valor do total. Em seguida, calcule a quantidade de laranjas dadas à vizinha e subtraia esse valor do que restou."
      }]
      }      };

    let currentQuestionIndex = 0;
    let userAnswers = {};
    const questionsArea = document.getElementById('questions-area');
    const progressBar = document.getElementById('progress-bar');
    const quizContainer = document.getElementById('quiz-container');
    const resultsArea = document.getElementById('results-area');

    function renderQuestion() {
        if (currentQuestionIndex >= data.simulado.questoes.length) {
            showResults();
            return;
        }

        const question = data.simulado.questoes[currentQuestionIndex];
        const questionHtml = `
            <div class="question-container active">
                <div class="question-info">Questão ${currentQuestionIndex + 1} de ${data.simulado.questoes.length} - ${question.disciplina}</div>
                <div class="enunciado">
                    ${question.enunciado}
                </div>
                <ul class="alternativas">
                    ${question.alternativas.map((alt, index) => `
                        <li class="alternativa-item" data-index="${index}">${alt}</li>
                    `).join('')}
                </ul>
                <button class="hint-button">Dica</button>
                <div class="hint-text">
                    <p>${question.dica}</p>
                </div>
                <div class="explanation">
                    <h4>Explicação:</h4>
                    <p>${question.explicacao}</p>
                </div>
                <div class="navigation-buttons">
                    <button id="prev-button" ${currentQuestionIndex === 0 ? 'disabled' : ''}>Voltar</button>
                    <button id="next-button">Avançar</button>
                </div>
            </div>
        `;

        questionsArea.innerHTML = questionHtml;
        updateProgressBar();
        addEventListeners();
        
        // Restore user's answer and explanation if already answered
        const savedAnswerIndex = userAnswers[currentQuestionIndex];
        if (savedAnswerIndex !== undefined) {
            const selectedItem = questionsArea.querySelector(`[data-index="${savedAnswerIndex}"]`);
            const isCorrect = selectedItem.textContent === question.resposta_correta;
            selectedItem.classList.add(isCorrect ? 'correct' : 'incorrect');
            if (questionsArea.querySelector('.explanation')) {
                questionsArea.querySelector('.explanation').style.display = 'block';
            }
            questionsArea.querySelectorAll('.alternativa-item').forEach(item => item.style.pointerEvents = 'none');
        }
    }

    function addEventListeners() {
        const alternativas = document.querySelectorAll('.alternativa-item');
        alternativas.forEach(item => {
            item.addEventListener('click', handleAnswerClick);
        });

        document.getElementById('prev-button').addEventListener('click', () => {
            currentQuestionIndex--;
            renderQuestion();
        });

        document.getElementById('next-button').addEventListener('click', () => {
            currentQuestionIndex++;
            renderQuestion();
        });

        document.querySelector('.hint-button').addEventListener('click', () => {
            const hintText = document.querySelector('.hint-text');
            hintText.style.display = hintText.style.display === 'block' ? 'none' : 'block';
        });
    }

    function handleAnswerClick(event) {
        const selectedItem = event.target;
        const question = data.simulado.questoes[currentQuestionIndex];
        const isCorrect = selectedItem.textContent === question.resposta_correta;
        
        // Save the user's answer
        userAnswers[currentQuestionIndex] = parseInt(selectedItem.dataset.index);

        selectedItem.classList.add(isCorrect ? 'correct' : 'incorrect');
        
        const explanation = document.querySelector('.explanation');
        explanation.style.display = 'block';

        // Disable other options
        const allAlternatives = document.querySelectorAll('.alternativa-item');
        allAlternatives.forEach(item => {
            item.style.pointerEvents = 'none';
        });

        // In review mode, show correct/incorrect on all options
        if (quizContainer.classList.contains('review-mode')) {
            const correctAlternative = questionsArea.querySelector(`[data-index="${question.alternativas.indexOf(question.resposta_correta)}"]`);
            if (correctAlternative) {
                correctAlternative.classList.add('correct');
            }
        }
    }

    function updateProgressBar() {
        const progress = ((currentQuestionIndex) / data.simulado.questoes.length) * 100;
        progressBar.style.width = `${progress}%`;
    }

    function showResults() {
        quizContainer.style.display = 'none';
        resultsArea.style.display = 'block';
        
        let correctCount = 0;
        data.simulado.questoes.forEach((question, index) => {
            const userAnswerIndex = userAnswers[index];
            if (userAnswerIndex !== undefined) {
                const userAnswer = question.alternativas[userAnswerIndex];
                if (userAnswer === question.resposta_correta) {
                    correctCount++;
                }
            }
        });

        const percentage = (correctCount / data.simulado.questoes.length) * 100;
        document.getElementById('correct-count').textContent = correctCount;
        document.getElementById('percentage').textContent = percentage.toFixed(2);
    }
    
    document.getElementById('review-button').addEventListener('click', () => {
        currentQuestionIndex = 0;
        resultsArea.style.display = 'none';
        quizContainer.style.display = 'block';
        quizContainer.classList.add('review-mode');
        renderQuestion();
        
        // Hide hint and explanation initially in review mode
        questionsArea.querySelector('.hint-text').style.display = 'none';
        questionsArea.querySelector('.explanation').style.display = 'block';
        
        // Disable answering in review mode, show the correct answer and user's choice
        const question = data.simulado.questoes[currentQuestionIndex];
        const userSelected = userAnswers[currentQuestionIndex];
        
        questionsArea.querySelectorAll('.alternativa-item').forEach(item => {
            item.style.pointerEvents = 'none';
            if (parseInt(item.dataset.index) === userSelected) {
                const isCorrect = item.textContent === question.resposta_correta;
                item.classList.add(isCorrect ? 'correct' : 'incorrect');
            }
        });
        
    });

    renderQuestion();

</script>

</body>
</html>"""

In [0]:
displayHTML(html)